# 08 — Official Deepnote Native 8-Task Benchmark & Difficulty Audit (`Official-8` vs `PageMantle-Ext`)

**Strictly Non-Interfering Standalone Extension**: Writes exclusively to `$DRIVE_ROOT/runs/deepnote_official_8/` and new filenames in `$DRIVE_ROOT/paper_artifacts/` (`tab_deepnote_official_8.tex`, `fig_deepnote_official_8.pdf`). Does **NOT** overwrite or modify any output from Notebooks `00`–`07`.

**What This Notebook Tests (100% Authentic Deepnote Official Benchmark)**:
1. Runs Deepnote's **8 REAL open-source official tasks** (`task-01`..`task-08` from `deepnote/agent-initiative-benchmark`) directly inside Deepnote's genuine 11-service `SlackWorker` runtime with all task-specific `delta/<service>/seed.json` overlays (Slack, PostHog, BigQuery, Notion, Stripe, Linear, HubSpot, Calendar, Analytics, Search Console, Snowflake).
2. Evaluates paired **`B0_official_direct`** (Deepnote's exact `assess_recent_slack.txt` prompt + full native tools) vs **`DIAL_Skill_official`** (Stage-1 Jev Triage + Stage-1.5 read-only Silent Recon over real enterprise services + Stage-2 Parallel Symmetric-Evidence Gate + Stage-3 routed execution via real Slack write tools `slack_reply_to_thread` / `slack_post_message` / `slack_send_dm`).
3. Grades ONLY actual delivered Slack messages (or silence) using Deepnote's **strict 0–10 integer rubric** (`evaluate_agent_run.txt`) including explicit `important_mistakes` zero-conditions — plus optional cross-family verification via `DeepSeek-V4.1-Flash`.
4. **Honest Benchmark Difficulty Audit**: Compares `Official-8` scores directly against our Notebook `04` `PageMantle-Ext` results to diagnose whether `PageMantle-Ext` is too easy and requires hardening.

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs/deepnote_official_8",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


In [ ]:
# === CELL 2: Bootstrap Multi-Provider Secrets & Configs (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import (
    describe_secret_sources,
    load_autodl_api_key,
    load_jev_api_key,
    load_openai_api_key,
)
from dial_skill.config import load_dial_skill_config, load_models_config

print(
    "Secret lookup diagnostics:",
    json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT, include_autodl=True), indent=2),
)
OPENAI_API_KEY = load_openai_api_key(drive_root=DRIVE_ROOT, required=True)
JEV_API_KEY = load_jev_api_key(drive_root=DRIVE_ROOT, required=True)
AUTODL_API_KEY = load_autodl_api_key(drive_root=DRIVE_ROOT, required=False)
AUTODL_BASE_URL = os.environ.get("AUTODL_BASE_URL", "https://www.autodl.art/api/v1")

models_cfg = load_models_config()
base_cfg = load_dial_skill_config()
print(f"OpenAI key loaded: {bool(OPENAI_API_KEY)} | Jev key loaded: {bool(JEV_API_KEY)} | AutoDL key loaded: {bool(AUTODL_API_KEY)}")


## Step 1 — Install Deepnote's Official Benchmark Runtime & Prepare Verified Dataset Snapshot

- Clones `https://github.com/deepnote/agent-initiative-benchmark.git` into `external/agent-initiative-benchmark` (already `.gitignore`d).
- Installs only the 4 lightweight runtime dependencies missing on Colab (`fastembed`, `hnswlib`, `ijson`, `pypdf`) without overwriting Colab's scientific stack (`numpy`/`pandas`/`scipy`), then installs `benchmark` in editable `--no-deps` mode.
- Prepares the verified PageMantle 11-service dataset into fast local Colab SSD (`/content/pagemantle_official`), **reusing the pinned archives already saved on Google Drive (`datasets/pagemantle`)** whenever the manifest SHA256 matches so there is zero 6 GB re-download.

In [ ]:
# === CELL 3: Clone/Install Deepnote Runtime & Prepare Verified Local Dataset ===
import hashlib
import shutil

DEEPNOTE_REPO_URL = "https://github.com/deepnote/agent-initiative-benchmark.git"
DEEPNOTE_REPO_DIR = (REPO_DIR / "external" / "agent-initiative-benchmark").resolve()
DEEPNOTE_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if not (DEEPNOTE_REPO_DIR / "tasks" / "index.json").is_file():
    !git clone --depth 1 {DEEPNOTE_REPO_URL} {DEEPNOTE_REPO_DIR}
else:
    print("Deepnote benchmark repo already present:", DEEPNOTE_REPO_DIR)

if IN_COLAB:
    !pip install -q fastembed hnswlib ijson pypdf
    !pip install -q --no-deps -e {DEEPNOTE_REPO_DIR}

_dn_src = str((DEEPNOTE_REPO_DIR / "src").resolve())
if _dn_src not in sys.path:
    sys.path.insert(0, _dn_src)

# Disk check on fast local SSD
OFFICIAL_DATA_DIR = Path("/content/pagemantle_official") if IN_COLAB else (DRIVE_ROOT / "pagemantle_official")
OFFICIAL_CACHE_DIR = Path("/content/pagemantle_official_cache") if IN_COLAB else (DRIVE_ROOT / "pagemantle_official_cache")
OFFICIAL_WORK_ROOT = Path("/content/pagemantle_official_work") if IN_COLAB else (DRIVE_ROOT / "pagemantle_official_work")
OFFICIAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(OFFICIAL_DATA_DIR.parent).free / 1e9
print(f"Free disk space at {OFFICIAL_DATA_DIR.parent}: {free_gb:.2f} GB")

from benchmark.dataset import Release, prepare_data

release = Release()
expected_manifest_sha = release.lock["dataset"]["manifest_sha256"]
drive_pm_source = DRIVE_ROOT / "datasets" / "pagemantle"
drive_manifest = drive_pm_source / "manifest.json"

use_drive_source = None
if drive_manifest.is_file():
    actual_sha = hashlib.sha256(drive_manifest.read_bytes()).hexdigest()
    has_all_tars = all((drive_pm_source / pkg["path"]).is_file() for pkg in release.packages.values())
    if actual_sha == expected_manifest_sha and has_all_tars and (drive_pm_source / "file-index.jsonl.gz").is_file():
        use_drive_source = drive_pm_source
        print(f"Matched pinned Deepnote archives on Google Drive ({drive_pm_source}) — extracting locally with ZERO 6GB re-download!")
    else:
        print(
            f"Drive datasets/pagemantle manifest mismatch or incomplete (sha_match={actual_sha == expected_manifest_sha}, "
            f"all_tars={has_all_tars}); falling back to HuggingFace download."
        )
else:
    print("Drive datasets/pagemantle/manifest.json not found; preparing directly from HuggingFace.")

prep_info = prepare_data(
    root=OFFICIAL_DATA_DIR,
    source=use_drive_source,
    cache_dir=OFFICIAL_CACHE_DIR,
    release=release,
)
print("Prepared official Deepnote dataset:", prep_info)


## Step 2 — Inspect All 8 Real Deepnote Official Tasks & Run Zero-Cost Runtime Smoke Test

Verifies every task package (`task-01`..`task-08`), displays their true categories, intents (`task-03` = `silence`, 7 others = `action`), delta services, and important-mistake counts, and performs one zero-API-cost `SlackWorker` boot test on `task-01`.

In [ ]:
# === CELL 4: Load Official-8 Catalog & Zero-Cost Live Runtime Smoke Check ===
import pandas as pd
from dial_skill.benchmarks.deepnote_official import (
    DeepnoteOfficialTaskSession,
    load_deepnote_official_catalog,
)

official_tasks = load_deepnote_official_catalog(DEEPNOTE_REPO_DIR)
catalog_df = pd.DataFrame(
    [
        {
            "task_id": t.task_id,
            "category": t.category,
            "skill": t.mapped_skill,
            "intent": t.intent,
            "delta_services": ", ".join(t.delta_services),
            "n_tools": len(t.tool_scope),
            "n_mistakes": len(t.important_mistakes),
            "title": t.title,
        }
        for t in official_tasks
    ]
)
display(catalog_df)
assert len(official_tasks) == 8, f"Expected 8 official tasks, got {len(official_tasks)}"

# Zero-API-cost smoke test on task-01 to confirm SlackWorker boots cleanly with overlays
async def _smoke_check_runtime():
    t0 = official_tasks[0]
    async with DeepnoteOfficialTaskSession(
        task=t0,
        data_dir=OFFICIAL_DATA_DIR,
        cache_dir=OFFICIAL_CACHE_DIR,
        work_root=OFFICIAL_WORK_ROOT,
    ) as sess:
        msgs = sess.initial_context.get("messages", [])
        channels = sorted({(m.get("channel") or {}).get("name", m.get("channel_id", "?")) for m in msgs})
        print(
            f"[SMOKE OK] {t0.task_id} booted in {sess.setup_seconds:.2f}s | "
            f"snapshot={sess.initial_context.get('snapshot_time')} | "
            f"recent_msgs={len(msgs)} across channels={channels} | "
            f"native_tools={len(sess.raw_tools)}"
        )

await _smoke_check_runtime()


## Step 3 — Experiment Configuration (Atomic JSONL Resume & Cost Control)

- **Default Backbones**: `["gpt-4.1-mini", "gpt-4.1"]` — directly paired with Notebook `04`'s `luna` (`gpt-4.1-mini`) and `terra` (`gpt-4.1`) scores on `PageMantle-Ext` for an exact apples-to-apples difficulty audit!
- You can optionally append `"gpt-4o"` or `"gpt-5.6-luna"` to `EVAL_MODELS` later; completed `(method, model_id, task_id)` episodes are skipped automatically on re-run.
- **Estimated Runtime & Cost**: ~8–15 min total for 2 models × 2 methods × 8 tasks = 32 episodes + strict dual-pass judging (~\$0.60–\$1.80 total on OpenAI).

In [ ]:
# === CELL 5: Configure Official-8 Run Knobs & Atomic Resume Helpers ===
RUN_DIR = DRIVE_ROOT / "runs" / "deepnote_official_8"
RUN_DIR.mkdir(parents=True, exist_ok=True)
EPISODES_PATH = RUN_DIR / "episodes_official_8.jsonl"
PRIMARY_JUDGMENTS_PATH = RUN_DIR / "judgments_official_8_primary.jsonl"
CROSS_JUDGMENTS_PATH = RUN_DIR / "judgments_official_8_cross_family.jsonl"
SUMMARY_PATH = RUN_DIR / "summary_deepnote_official_8.json"

# Directly matches Notebook 04's gpt-4.1-mini (luna) and gpt-4.1 (terra) for Official-8 vs PageMantle-Ext audit
EVAL_MODELS = ["gpt-4.1-mini", "gpt-4.1"]
PRIMARY_JUDGE_MODEL = "gpt-4.1"
RUN_CROSS_FAMILY_JUDGE = bool(AUTODL_API_KEY)
AUTODL_JUDGE_MODEL = "DeepSeek-V4.1-Flash"

SELECTED_TASK_IDS = None  # None = all 8 tasks (task-01..task-08)
RECON_MAX_CALLS = 8
RECON_MAX_TURNS = 5
STAGE3_MAX_TURNS = 18
DIRECT_MAX_TURNS = 24
FORCE_RERUN = False

selected_tasks = (
    [t for t in official_tasks if t.task_id in set(SELECTED_TASK_IDS)]
    if SELECTED_TASK_IDS
    else list(official_tasks)
)

def load_jsonl_index(path: Path, key_fn):
    records = {}
    if not path.is_file():
        return records
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        records[key_fn(obj)] = obj
    return records

def append_jsonl_record(path: Path, obj: dict) -> None:
    with path.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(obj, ensure_ascii=False) + "\n")
        fh.flush()
        os.fsync(fh.fileno())

print(
    f"Configured {len(EVAL_MODELS)} models x 2 methods x {len(selected_tasks)} tasks = "
    f"{len(EVAL_MODELS) * 2 * len(selected_tasks)} episodes | "
    f"Primary Judge={PRIMARY_JUDGE_MODEL} | Cross-Family Judge={AUTODL_JUDGE_MODEL if RUN_CROSS_FAMILY_JUDGE else 'DISABLED'}"
)


## Step 4 — Run Paired `B0_official_direct` vs `DIAL_Skill_official` on All 8 Real Tasks

Each `(method, model_id, task_id)` runs inside a brand-new isolated `DeepnoteOfficialTaskSession` (`SlackWorker`) with all real `delta/<service>/seed.json` overlays, and persists immediately to Google Drive (`episodes_official_8.jsonl`).

In [ ]:
# === CELL 6: Execute Paired Official Direct vs DIAL-Skill Across All 8 Real Tasks ===
from dial_skill.benchmarks.deepnote_official import (
    OfficialDialSkillAgent,
    OfficialDirectAgent,
    OfficialEpisodeResult,
)
from dial_skill.core.skill_store import SkillStore
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

api_cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache_official8.sqlite")
openai_client = OpenAIClient(api_key=OPENAI_API_KEY, models_cfg=models_cfg, cache=api_cache)
triage_client = TriageClient(
    models_cfg=models_cfg,
    cache=api_cache,
    openai_client=openai_client,
    luna_model="gpt-4.1-mini",
    jev_api_key=JEV_API_KEY,
)
skill_store = SkillStore.from_json_file()

def _ep_key(obj: dict) -> tuple[str, str, str]:
    return (str(obj["method"]), str(obj["model_id"]), str(obj["task_id"]))

if FORCE_RERUN and EPISODES_PATH.exists():
    EPISODES_PATH.unlink()

completed_eps = load_jsonl_index(EPISODES_PATH, _ep_key)
print(f"Loaded {len(completed_eps)} already-completed episodes from {EPISODES_PATH.name}")

async def run_all_official_8_episodes():
    for model_id in EVAL_MODELS:
        direct_agent = OfficialDirectAgent(
            openai_client=openai_client,
            model=model_id,
            method_name="B0_official_direct",
            max_turns=DIRECT_MAX_TURNS,
        )
        dial_agent = OfficialDialSkillAgent(
            triage_client=triage_client,
            openai_client=openai_client,
            gate_model=model_id,
            executor_model=model_id,
            skill_store=skill_store,
            cfg=base_cfg,
            method_name="DIAL_Skill_official",
            recon_max_calls=RECON_MAX_CALLS,
            recon_max_turns=RECON_MAX_TURNS,
            stage3_max_turns=STAGE3_MAX_TURNS,
        )

        # Canonical model_id key stored by OfficialDialSkillAgent is f"{model}+{model}"
        agent_specs = [
            ("B0_official_direct", model_id, direct_agent),
            ("DIAL_Skill_official", f"{model_id}+{model_id}", dial_agent),
        ]

        print(f"\n{'=' * 78}\nRunning Deepnote Official-8 for backbone: {model_id}\n{'=' * 78}")
        for task in selected_tasks:
            for method_name, stored_model_id, agent in agent_specs:
                key = (method_name, stored_model_id, task.task_id)
                if key in completed_eps:
                    ep_prev = completed_eps[key]
                    print(
                        f"  [SKIP] {task.task_id} ({task.category:10s} | intent={task.intent:7s}) | "
                        f"{method_name:20s} -> pred={ep_prev['predicted_action']:9s} "
                        f"deliv={len(ep_prev.get('delivered_messages', []))} "
                        f"reads={ep_prev.get('read_calls_used', 0)} stop={ep_prev.get('stop_reason')}"
                    )
                    continue

                async with DeepnoteOfficialTaskSession(
                    task=task,
                    data_dir=OFFICIAL_DATA_DIR,
                    cache_dir=OFFICIAL_CACHE_DIR,
                    work_root=OFFICIAL_WORK_ROOT,
                ) as session:
                    ep: OfficialEpisodeResult = await agent.run_task(session)

                ep_dict = ep.model_dump(mode="json")
                append_jsonl_record(EPISODES_PATH, ep_dict)
                completed_eps[key] = ep_dict

                gate_info = ""
                if ep.gate is not None:
                    gate_info = (
                        f" | skill={ep.gate.selected_skill} tier={ep.gate.final_tier.value} "
                        f"adv={ep.gate.advocate_score:.2f} skep={ep.gate.skeptic_score:.2f}"
                    )
                print(
                    f"  [DONE] {task.task_id} ({task.category:10s} | intent={task.intent:7s}) | "
                    f"{method_name:20s} -> pred={ep.predicted_action:9s} "
                    f"deliv={len(ep.delivered_messages)} reads={ep.read_calls_used:2d} writes={ep.write_calls_used} "
                    f"sec={ep.model_elapsed_sec:5.1f}s cost=${ep.total_cost_usd:.4f} stop={ep.stop_reason}{gate_info}"
                )

await run_all_official_8_episodes()
print(
    f"\nExecution complete! Total episodes stored: {len(completed_eps)} | "
    f"Session OpenAI API cost so far: ${openai_client.usage.total_cost_usd:.4f} | "
    f"Jev cost: ${triage_client.total_cost_usd:.4f}"
)


## Step 5 — Strict Deepnote-Faithful Blinded Dual-Judge & Cross-Family Agreement

Grades every episode strictly on **actual delivered Slack messages (or silence)** against Deepnote's authoritative `task.json` (`notes`, `evidence`, `reference.example_messages`, `reference.important_mistakes`):
- Enforces Deepnote's explicit zero conditions (unwarranted intervention on `task-03` = `0`; silence on action task = `0`; triggered serious mistake = `0`).
- Runs 2 order-swapped blinded passes on `gpt-4.1`, plus cross-family grading on `DeepSeek-V4.1-Flash` (via AutoDL) to verify zero self-preference bias.

In [ ]:
# === CELL 7: Grade All Official-8 Episodes with Strict Deepnote Judge (+ Cross-Family Check) ===
from dial_skill.benchmarks.deepnote_official import (
    DeepnoteOfficialJudge,
    DeepnoteRubricJudgment,
)
from dial_skill.eval.judge import compute_cohens_kappa, compute_spearman_rho

task_by_id = {t.task_id: t for t in official_tasks}

def _j_key(obj: dict) -> tuple[str, str, str]:
    return (str(obj["method"]), str(obj["model_id"]), str(obj["task_id"]))

if FORCE_RERUN:
    for p in (PRIMARY_JUDGMENTS_PATH, CROSS_JUDGMENTS_PATH):
        if p.exists():
            p.unlink()

primary_judgments_raw = load_jsonl_index(PRIMARY_JUDGMENTS_PATH, _j_key)
primary_judge = DeepnoteOfficialJudge(
    openai_client=openai_client,
    judge_model=PRIMARY_JUDGE_MODEL,
    num_passes=2,
    seed=42,
)

async def run_primary_judging():
    for key, ep_dict in sorted(completed_eps.items()):
        if key in primary_judgments_raw:
            continue
        ep = OfficialEpisodeResult.model_validate(ep_dict)
        task = task_by_id[ep.task_id]
        j = await primary_judge.grade_episode(task=task, episode=ep)
        j_dict = j.model_dump(mode="json")
        append_jsonl_record(PRIMARY_JUDGMENTS_PATH, j_dict)
        primary_judgments_raw[key] = j_dict
        zero_tag = " [SERIOUS_ZERO]" if j.serious_mistake_zero else ""
        print(
            f"  [JUDGE-1] {ep.task_id} | {ep.method:20s} ({ep.model_id}) -> "
            f"overall={j.overall_score:4.1f} (IoS={j.intervention_or_silence:.1f}, "
            f"CE={j.correctness_and_evidence}, Man={j.manner_of_intervention}, Aud={j.audience_and_location}){zero_tag}"
        )

await run_primary_judging()
print(f"Primary judging complete ({len(primary_judgments_raw)} graded). OpenAI cumulative cost: ${openai_client.usage.total_cost_usd:.4f}")

cross_judgments_raw = load_jsonl_index(CROSS_JUDGMENTS_PATH, _j_key)
cross_family_agreement = None

if RUN_CROSS_FAMILY_JUDGE and AUTODL_API_KEY:
    autodl_client = OpenAIClient(
        api_key=AUTODL_API_KEY,
        models_cfg=models_cfg,
        cache=api_cache,
        base_url=AUTODL_BASE_URL,
        provider_name="autodl",
    )
    cross_judge = DeepnoteOfficialJudge(
        openai_client=autodl_client,
        judge_model=AUTODL_JUDGE_MODEL,
        num_passes=1,
        seed=101,
    )

    async def run_cross_judging():
        for key, ep_dict in sorted(completed_eps.items()):
            if key in cross_judgments_raw:
                continue
            ep = OfficialEpisodeResult.model_validate(ep_dict)
            task = task_by_id[ep.task_id]
            try:
                j = await cross_judge.grade_episode(task=task, episode=ep)
                j_dict = j.model_dump(mode="json")
                append_jsonl_record(CROSS_JUDGMENTS_PATH, j_dict)
                cross_judgments_raw[key] = j_dict
            except Exception as exc:
                print(f"  [WARN] Cross-family judge failed on {key}: {type(exc).__name__}: {exc}")

    await run_cross_judging()

    paired_keys = sorted(set(primary_judgments_raw) & set(cross_judgments_raw))
    if len(paired_keys) >= 4:
        s_a = [float(primary_judgments_raw[k]["overall_score"]) for k in paired_keys]
        s_b = [float(cross_judgments_raw[k]["overall_score"]) for k in paired_keys]
        rho_info = compute_spearman_rho(s_a, s_b)
        b_a = ["zero" if x <= 0.5 else ("mid" if x < 7.0 else "high") for x in s_a]
        b_b = ["zero" if x <= 0.5 else ("mid" if x < 7.0 else "high") for x in s_b]
        kappa = compute_cohens_kappa(b_a, b_b)
        mae = round(sum(abs(x - y) for x, y in zip(s_a, s_b)) / len(s_a), 4)
        cross_family_agreement = {
            "n_paired": len(paired_keys),
            "primary_judge": PRIMARY_JUDGE_MODEL,
            "secondary_judge": AUTODL_JUDGE_MODEL,
            "spearman_rho": rho_info["rho"],
            "spearman_p": rho_info["p_value"],
            "cohens_kappa_3band": kappa,
            "mae": mae,
        }
        print("\nCross-Family Judge Agreement (Official-8):", json.dumps(cross_family_agreement, indent=2))
else:
    print("Cross-family AutoDL judge skipped (AUTODL_API_KEY not found or disabled).")


## Step 6 — Per-Task Diagnostic Breakdown & Honest Difficulty Audit (`Official-8` vs `PageMantle-Ext`)

This is the critical diagnostic cell:
1. Shows **aggregate metrics** for `B0_official_direct` vs `DIAL_Skill_official` on Deepnote's 8 real tasks.
2. Shows every single task (`task-01`..`task-08`) side-by-side so you can inspect `task-03` (cross-service silence trap), `task-08` (partial-answer PII guard), and any triggered `important_mistakes`.
3. Automatically loads Notebook `04`'s read-only `PageMantle-Ext` summary and computes the exact **difficulty gap ($\Delta = \text{Official-8} - \text{PageMantle-Ext}$)** so we can objectively decide whether `PageMantle-Ext` is too easy and needs hardening!

In [ ]:
# === CELL 8: Full Diagnostic Breakdown & Official-8 vs PageMantle-Ext Difficulty Audit ===
from dial_skill.benchmarks.deepnote_official import summarize_official_8_results

def _canonical_backbone(model_id: str) -> str:
    return model_id.split("+")[0]

# 1. Aggregate Table
agg_rows = []
summary_by_group = {}
for model_id in EVAL_MODELS:
    for method_name, stored_mid in [
        ("B0_official_direct", model_id),
        ("DIAL_Skill_official", f"{model_id}+{model_id}"),
    ]:
        eps = [
            OfficialEpisodeResult.model_validate(completed_eps[(method_name, stored_mid, t.task_id)])
            for t in selected_tasks
            if (method_name, stored_mid, t.task_id) in completed_eps
        ]
        jmap = {
            t.task_id: DeepnoteRubricJudgment.model_validate(primary_judgments_raw[(method_name, stored_mid, t.task_id)])
            for t in selected_tasks
            if (method_name, stored_mid, t.task_id) in primary_judgments_raw
        }
        stats = summarize_official_8_results(eps, jmap)
        summary_by_group[f"{method_name}__{model_id}"] = stats
        agg_rows.append({"backbone": model_id, "method": method_name, **stats})

agg_df = pd.DataFrame(agg_rows)
print("=== 1. AGGREGATE DEEPNOTE OFFICIAL-8 RESULTS (Strict Deepnote 0-10 Rubric) ===")
display(agg_df)

# 2. Per-Task Side-by-Side Table
per_task_rows = []
for model_id in EVAL_MODELS:
    for t in selected_tasks:
        d_key = ("B0_official_direct", model_id, t.task_id)
        s_key = ("DIAL_Skill_official", f"{model_id}+{model_id}", t.task_id)
        d_ep = completed_eps.get(d_key, {})
        s_ep = completed_eps.get(s_key, {})
        d_j = primary_judgments_raw.get(d_key, {})
        s_j = primary_judgments_raw.get(s_key, {})
        per_task_rows.append(
            {
                "backbone": model_id,
                "task_id": t.task_id,
                "category": t.category,
                "intent": t.intent,
                "Direct_Overall": d_j.get("overall_score"),
                "DIAL_Overall": s_j.get("overall_score"),
                "Delta_DIAL_vs_Direct": (
                    round(float(s_j["overall_score"]) - float(d_j["overall_score"]), 2)
                    if ("overall_score" in s_j and "overall_score" in d_j)
                    else None
                ),
                "Direct_Act": d_ep.get("predicted_action"),
                "DIAL_Act": s_ep.get("predicted_action"),
                "Direct_Reads": d_ep.get("read_calls_used"),
                "DIAL_Reads": s_ep.get("read_calls_used"),
                "Direct_Mistakes": "; ".join(d_j.get("triggered_important_mistakes", []))[:80],
                "DIAL_Mistakes": "; ".join(s_j.get("triggered_important_mistakes", []))[:80],
            }
        )

per_task_df = pd.DataFrame(per_task_rows)
print("\n=== 2. PER-TASK BREAKDOWN (task-01 .. task-08) ===")
display(per_task_df)

# 3. Honest Difficulty Audit: Official-8 vs PageMantle-Ext (Read-Only from Notebook 04)
pm_ext_candidates = [
    DRIVE_ROOT / "runs" / "main_pagemantle" / "full" / "pre_evolution_metrics.json",
    DRIVE_ROOT / "runs" / "main_pagemantle" / "pilot" / "pre_evolution_metrics.json",
]
pm_ext_summary_path = next((p for p in pm_ext_candidates if p.is_file()), pm_ext_candidates[0])
difficulty_audit = []
if pm_ext_summary_path.is_file():
    pm_ext = json.loads(pm_ext_summary_path.read_text(encoding="utf-8"))
    print(f"Loaded Notebook 04 PageMantle-Ext metrics (read-only): {pm_ext_summary_path}")
    ext_map = {
        ("B0_official_direct", "gpt-4.1-mini"): ("B0_direct_luna", pm_ext.get("B0_direct_luna")),
        ("DIAL_Skill_official", "gpt-4.1-mini"): ("DIAL_Skill_v0_luna", pm_ext.get("DIAL_Skill_v0_luna")),
        ("B0_official_direct", "gpt-4.1"): ("B0_direct_terra", pm_ext.get("B0_direct_terra")),
        ("DIAL_Skill_official", "gpt-4.1"): ("DIAL_Skill_v0_terra", pm_ext.get("DIAL_Skill_v0_terra")),
    }
    for (off_method, model_id), (ext_name, ext_stats) in ext_map.items():
        off_stats = summary_by_group.get(f"{off_method}__{model_id}")
        if not off_stats or not ext_stats:
            continue
        off_score = float(off_stats.get("overall_mean", 0.0))
        ext_score = float(ext_stats.get("mean_overall_score", ext_stats.get("overall_score", 0.0)))
        gap = round(off_score - ext_score, 3)
        difficulty_audit.append(
            {
                "backbone": model_id,
                "method": off_method,
                "Official_8_Score": round(off_score, 3),
                "PageMantle_Ext_Score": round(ext_score, 3),
                "Gap_Official_minus_Ext": gap,
                "Official_8_SilAcc": off_stats.get("silence_correct"),
                "Ext_SilAcc": ext_stats.get("silence_accuracy"),
                "Official_8_AvgReads": off_stats.get("avg_read_calls"),
                "Serious_Zeros_Official_8": off_stats.get("serious_mistake_zeros"),
            }
        )
    audit_df = pd.DataFrame(difficulty_audit)
    print("\n=== 3. HONEST DIFFICULTY AUDIT: DEEPNOTE OFFICIAL-8 vs PAGEMANTLE-EXT (Notebook 04) ===")
    display(audit_df)
    large_gaps = [r for r in difficulty_audit if abs(float(r["Gap_Official_minus_Ext"])) >= 1.5]
    if large_gaps:
        print(
            "\n[DIAGNOSTIC VERDICT] SIGNIFICANT DIFFICULTY GAP DETECTED (|Official-8 - PageMantle-Ext| >= 1.5)!\n"
            "This confirms PageMantle-Ext is noticeably easier than Deepnote's real Official-8 — we SHOULD inspect "
            "the failure modes below and harden PageMantle-Ext (multi-channel distractors, cross-service silence traps, "
            "strict important_mistakes zero-conditions, and partial-answer guard cases)."
        )
    else:
        print(
            "\n[DIAGNOSTIC VERDICT] Official-8 and PageMantle-Ext scores are within 1.5 points across paired settings. "
            "Inspect per-task mistakes (especially task-03 silence trap & task-08 guard) below for fine-grained validation."
        )
else:
    print(f"Note: {pm_ext_summary_path} not found locally; skipping automatic PageMantle-Ext diff.")

# 4. Qualitative Inspector (task-03 silence trap, task-08 guard trap, and any score < 7)
print("\n=== 4. QUALITATIVE CASE INSPECTOR (task-03, task-08, & any episode < 7.0) ===")
for model_id in EVAL_MODELS:
    for t in selected_tasks:
        for method_name, stored_mid in [
            ("B0_official_direct", model_id),
            ("DIAL_Skill_official", f"{model_id}+{model_id}"),
        ]:
            key = (method_name, stored_mid, t.task_id)
            if key not in primary_judgments_raw:
                continue
            j = primary_judgments_raw[key]
            ep = completed_eps[key]
            if t.task_id in ("task-03", "task-08") or float(j["overall_score"]) < 7.0:
                print(
                    f"\n--- {t.task_id} ({t.category}, intent={t.intent}) | {method_name} [{model_id}] | "
                    f"overall={j['overall_score']} | pred={ep['predicted_action']} | stop={ep['stop_reason']} ---"
                )
                print("Judge Overall Reason:", j.get("justifications", {}).get("overall", ""))
                if j.get("triggered_important_mistakes"):
                    print("Triggered Mistakes:", j["triggered_important_mistakes"])
                for idx, dm in enumerate(ep.get("delivered_messages", []), 1):
                    print(
                        f"  Delivered #{idx} [{dm['tool']} -> {dm.get('channel_name') or dm.get('destination_user_id')}]: "
                        f"{dm['text'][:420]}"
                    )


## Step 7 — Export Standalone Paper Artifacts (`tab_deepnote_official_8.tex` & `fig_deepnote_official_8.pdf`)

Writes ONLY new incremental files into `$DRIVE_ROOT/runs/deepnote_official_8/` and `$DRIVE_ROOT/paper_artifacts/` without altering any Notebook `00`–`07` tables or figures.

In [ ]:
# === CELL 9: Export Standalone JSON Summary, LaTeX Table & Diagnostic Figure ===
import matplotlib.pyplot as plt
import numpy as np

summary_payload = {
    "benchmark": "deepnote_official_8",
    "repo": "deepnote/agent-initiative-benchmark",
    "tasks": [t.task_id for t in selected_tasks],
    "eval_models": EVAL_MODELS,
    "primary_judge": PRIMARY_JUDGE_MODEL,
    "cross_family_agreement": cross_family_agreement,
    "aggregate": summary_by_group,
    "per_task": per_task_rows,
    "difficulty_audit_vs_pagemantle_ext": difficulty_audit,
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved summary JSON:", SUMMARY_PATH)

# Export standalone LaTeX table
tex_lines = [
    "% Auto-generated by Notebook 08 (Deepnote Official-8 Benchmark) — non-interfering standalone table",
    "\\begin{table}[t]",
    "\\centering",
    "\\caption{Evaluation on Deepnote's 8 Real Native Official Tasks (\\texttt{task-01}--\\texttt{task-08}, 11-service runtime with task deltas, strict Deepnote 0--10 rubric with \\texttt{important\\_mistakes} zero-conditions).}",
    "\\label{tab:deepnote_official_8}",
    "\\small",
    "\\begin{tabular}{llccccccc}",
    "\\toprule",
    "Backbone & Method & Overall & Int/Sil & Content & Audience & SilAcc & ActRec & Avg Reads \\\\",
    "\\midrule",
]
for row in agg_rows:
    m_label = "Direct (Deepnote Official)" if row["method"] == "B0_official_direct" else "DIAL-Skill (Ours)"
    ce_str = f"{row['correctness_and_evidence_mean']:.2f}" if row["correctness_and_evidence_mean"] is not None else "--"
    al_str = f"{row['audience_and_location_mean']:.2f}" if row["audience_and_location_mean"] is not None else "--"
    tex_lines.append(
        f"{row['backbone']} & {m_label} & {row['overall_mean']:.2f} & "
        f"{row['intervention_or_silence_mean']:.2f} & {ce_str} & {al_str} & "
        f"{row['silence_correct']} & {row['action_correct']} & {row['avg_read_calls']:.1f} \\\\"
    )
tex_lines += [
    "\\bottomrule",
    "\\end{tabular}",
    "\\end{table}",
]
tex_path = DRIVE_ROOT / "paper_artifacts" / "tables" / "tab_deepnote_official_8.tex"
tex_path.write_text("\n".join(tex_lines) + "\n", encoding="utf-8")
print("Saved LaTeX table:", tex_path)

# Export standalone 2-panel diagnostic PDF figure
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), dpi=200)

# Panel A: Per-task scores on Official-8 (using strongest backbone, e.g. gpt-4.1 if present else first)
focus_model = "gpt-4.1" if "gpt-4.1" in EVAL_MODELS else EVAL_MODELS[0]
sub_df = per_task_df[per_task_df["backbone"] == focus_model].copy()
x = np.arange(len(sub_df))
w = 0.36
axes[0].bar(x - w / 2, sub_df["Direct_Overall"].fillna(0.0), width=w, label="Direct (Official)", color="#6B7280")
axes[0].bar(x + w / 2, sub_df["DIAL_Overall"].fillna(0.0), width=w, label="DIAL-Skill (Ours)", color="#2563EB")
axes[0].set_xticks(x)
axes[0].set_xticklabels(
    [f"{r.task_id}\n({r.category[:5]})" for r in sub_df.itertuples()],
    fontsize=8.5,
)
axes[0].set_ylim(0, 11.0)
axes[0].set_ylabel("Strict Deepnote Overall Score (0-10)")
axes[0].set_title(f"(a) Deepnote Official-8 Per-Task Scores ({focus_model})")
axes[0].grid(axis="y", linestyle="--", alpha=0.35)
axes[0].legend(frameon=False)

# Panel B: Official-8 vs PageMantle-Ext Difficulty Comparison
if difficulty_audit:
    aud_df = pd.DataFrame(difficulty_audit)
    labels = [
        f"{'DIAL' if 'DIAL' in r.method else 'Direct'}\n({r.backbone})"
        for r in aud_df.itertuples()
    ]
    xb = np.arange(len(aud_df))
    axes[1].bar(xb - w / 2, aud_df["PageMantle_Ext_Score"], width=w, label="PageMantle-Ext (40t)", color="#93C5FD")
    axes[1].bar(xb + w / 2, aud_df["Official_8_Score"], width=w, label="Deepnote Official-8 (Real)", color="#DC2626")
    axes[1].set_xticks(xb)
    axes[1].set_xticklabels(labels, fontsize=8.5)
    axes[1].set_ylim(0, 11.0)
    axes[1].set_ylabel("Mean Overall Score (0-10)")
    axes[1].set_title("(b) Benchmark Difficulty Audit: Official-8 vs PageMantle-Ext")
    axes[1].grid(axis="y", linestyle="--", alpha=0.35)
    axes[1].legend(frameon=False)
else:
    xb = np.arange(len(agg_df))
    axes[1].bar(xb, agg_df["overall_mean"], color=["#6B7280", "#2563EB"] * len(EVAL_MODELS))
    axes[1].set_xticks(xb)
    axes[1].set_xticklabels([f"{r.backbone}\n{r.method[:6]}" for r in agg_df.itertuples()], fontsize=8.5)
    axes[1].set_ylim(0, 11.0)
    axes[1].set_title("(b) Deepnote Official-8 Mean Overall Scores")
    axes[1].grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()
fig_path = DRIVE_ROOT / "paper_artifacts" / "figures" / "fig_deepnote_official_8.pdf"
fig.savefig(fig_path, bbox_inches="tight")
plt.show()
print("Saved diagnostic figure:", fig_path)


In [ ]:
# === CELL 10: Optional Auto-Disconnect Colab Runtime (Saves Compute Units) ===
AUTO_DISCONNECT_COLAB = False  # Set True for unattended runs

import time

if IN_COLAB and AUTO_DISCONNECT_COLAB:
    print("Flushing Drive buffers and disconnecting Colab runtime in 5 seconds...")
    os.sync()
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Notebook 08 complete! Runtime left active (set AUTO_DISCONNECT_COLAB=True to auto-stop).")
